<h1><b>Grouping and Sorting</b></h1>
<span>Scale up your level of insight. The more complex the dataset, the more this matters</span>

# <h2 id="Introduction">Introduction</h2>

<p>Maps allow us to transform data in a DataFrame or Series one value at a time for an entire column. However, often we want to group our data, and then do something specific to the group the data is in.</p>

<p>As you'll learn, we do this with the <code>groupby()</code> operation.  We'll also cover some additional topics, such as more complex ways to index your DataFrames, along with how to sort your data.</p>

# <h2 id="Groupwise-analysis">Groupwise analysis</h2>

<p>One function we've been using heavily thus far is the <code>value_counts()</code> function. We can replicate what <code>value_counts()</code> does by doing the following:</p>

In [2]:
import pandas as pd
pd.set_option("display.max_rows", 5)

In [3]:
reviews = pd.read_csv("https://gist.githubusercontent.com/clairehq/79acab35be50eaf1c383948ed3fd1129/raw/407a02139ae1e134992b90b4b2b8c329b3d73a6a/winemag-data-130k-v2.csv")

In [4]:
reviews.groupby('points').count()

,Unnamed: 0,country,description,designation,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
points,,,,,,,,,,,,,
80,155,155,155,95,155,155,127,49,110,110,155,155,155
81,305,305,305,171,298,305,259,117,205,200,305,305,305
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,15,15,15,14,15,15,14,5,10,10,15,15,15
100,8,8,8,6,8,8,7,1,5,5,8,8,8


<p><code>groupby()</code> created a group of reviews which allotted the same point values to the given wines. Then, for each of these groups, we grabbed the <code>points()</code> column and counted how many times it appeared.  <code>value_counts()</code> is just a shortcut to this <code>groupby()</code> operation.</p>

<p>We can use any of the summary functions we've used before with this data. For example, to get the cheapest wine in each point value category, we can do the following:</p>

In [5]:
reviews.groupby('points').price.min()

,price
points,
80,5.0
81,5.0
...,...
99,75.0
100,150.0


<p>You can think of each group we generate as being a slice of our DataFrame containing only data with values that match. This DataFrame is accessible to us directly using the <code>apply()</code> method, and we can then manipulate the data in any way we see fit. For example, here's one way of selecting the name of the first wine reviewed from each winery in the dataset:</p>

In [7]:
reviews.groupby('winery').apply(lambda df: df.title.iloc[0], include_groups=False)

,0
winery,
1+1=3,1+1=3 NV Rosé Sparkling (Cava)
10 Knots,10 Knots 2010 Viognier (Paso Robles)
...,...
àMaurice,àMaurice 2013 Fred Estate Syrah (Walla Walla V...
Štoka,Štoka 2009 Izbrani Teran (Kras)


<p>For even more fine-grained control, you can also group by more than one column. For an example, here's how we would pick out the best wine by country <em>and</em> province:</p>

In [8]:
reviews.groupby(['country', 'province']).apply(lambda df: df.loc[df.points.idxmax()])

/tmp/ipython-input-1865732994.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reviews.groupby(['country', 'province']).apply(lambda df: df.loc[df.points.idxmax()])


Unnamed: 0    country  \
country   province                                  
Argentina Mendoza Province        8869  Argentina   
          Other                   4633  Argentina   
...                                ...        ...   
Uruguay   San Jose               39898    Uruguay   
          Uruguay                39361    Uruguay   

                                                                  description  \
country   province                                                              
Argentina Mendoza Province  If you love massive Argentine reds with purity...   
          Other             This single-vineyard Malbec blend from vineyar...   
...                                                                       ...   
Uruguay   San Jose          Baked, sweet, heavy aromas turn earthy with ti...   
          Uruguay           Cherry and berry aromas are ripe, healthy and ...   

                                                                  designation  \
country   province                                                              
Argentina Mendoza Province  Finca Pedregal Single Vineyard Barrancas Maipú...   
          Other                                                  Chañar Punco   
...                                                                       ...   
Uruguay   San Jose                                   El Preciado Gran Reserva   
          Uruguay                                   Blend 002 Limited Edition   

                            points  price          province          region_1  \
country   province                                                              
Argentina Mendoza Province      95   74.0  Mendoza Province           Mendoza   
          Other                 94   68.0             Other  Calchaquí Valley   
...                            ...    ...               ...               ...   
Uruguay   San Jose              87   50.0          San Jose               NaN   
          Uruguay               91   22.0           Uruguay               NaN   

                           region_2        taster_name taster_twitter_handle  \
country   province                                                             
Argentina Mendoza Province      NaN  Michael Schachner           @wineschach   
          Other                 NaN  Michael Schachner           @wineschach   
...                             ...                ...                   ...   
Uruguay   San Jose              NaN  Michael Schachner           @wineschach   
          Uruguay               NaN  Michael Schachner           @wineschach   

                                                                        title  \
country   province                                                              
Argentina Mendoza Province  Pascual Toso 2014 Finca Pedregal Single Vineya...   
          Other             El Esteco 2013 Chañar Punco Red (Calchaquí Val...   
...                                                                       ...   
Uruguay   San Jose          Castillo Viejo 2005 El Preciado Gran Reserva R...   
          Uruguay           Narbona NV Blend 002 Limited Edition Tannat-Ca...   

                                              variety          winery  
country   province                                                     
Argentina Mendoza Province  Cabernet Sauvignon-Malbec    Pascual Toso  
          Other                             Red Blend       El Esteco  
...                                               ...             ...  
Uruguay   San Jose                          Red Blend  Castillo Viejo  
          Uruguay               Tannat-Cabernet Franc         Narbona  

[385 rows x 14 columns]

<p>Another <code>groupby()</code> method worth mentioning is <code>agg()</code>, which lets you run a bunch of different functions on your DataFrame simultaneously. For example, we can generate a simple statistical summary of the dataset as follows:</p>

In [9]:
reviews.groupby(['country']).price.agg([len, min, max])

/tmp/ipython-input-4122224884.py:1: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  reviews.groupby(['country']).price.agg([len, min, max])
/tmp/ipython-input-4122224884.py:1: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  reviews.groupby(['country']).price.agg([len, min, max])


,len,min,max
country,,,
Argentina,1907,4.0,230.0
Armenia,1,14.0,14.0
...,...,...,...
Ukraine,5,6.0,10.0
Uruguay,61,10.0,120.0


<p>Effective use of <code>groupby()</code> will allow you to do lots of really powerful things with your dataset.</p>